<!-- .<h1 style="color:MediumSeaGreen;" align="center" class="tocSkip"> <center>Complex Numbers</center> </h1>  -->
<center><font size=6 color='MediumSeaGreen'><b>Introduction to quantum simulation</b> </font></center>

<h1 style="color:MediumSeaGreen;"> Summary </h1> 
<!-- <h1 style="color:MediumSeaGreen;" align='middle'> Summary </h1>  -->

- What is quantum simulation?
- Different aspects of quantum simulation
- Phase estimation algorithm
- Getting approximate wave function from classical computers.
- Mapping the electronic Hamiltonian to qubit Hamiltonian.
- Trotter suzuki approximation
- Phase estimation algorithm (PEA)
- Variational quantum eigensolver (VQE)

<h1 style="color:MediumSeaGreen;"> What is quantum simulation? </h1> 

World at the microscopic scales behave according to the rules of quantum mechanics. If we have a system of particles then their behavior is governed by Schrodinger's equation which is given by:

$$i\frac{\partial \Psi}{\partial t} (\vec{\mathbf{x}},\vec{\omega},t) = H(\vec{\mathbf{x}}) \Psi(\vec{\mathbf{x}},\vec{\omega},t)$$

with

$$H(\vec{\mathbf{x}})= \left(- \sum_i^N \left[\frac{\nabla_{x_i}^2}{2} +V_{ext}(x_i) +\frac{1}{2}\sum_{j}^{N} \frac{1}{|x_i-x_j|}\right] \right)$$

Here $\vec{\mathbf{x}}=(x_1,x_2,...x_N)$ represents the spatial coordinates of all $N$ electrons and $\vec{\omega}=(\omega_1,...\omega_N)$ represents their spin coordinates with $\omega_j=\pm \frac{1}{2}$.

In order to understand the dynamics of the quantum system we are required to solve the above equation. So, the goal of quantum simulation is to solve for the eigenfunctions of the above equation. As you can guess by
observing the equation that analytical solution to the above equation is almost impossible. In fact there are only a handful of quantum systems for which the analytical solution is known. For all the other systems, we need to use numerical methods to solve for the solution. How do the algorithms used to find the solution scale? It turns out the simulation for Schrodinger's equation is computationally ineffecient, meaning that the computational resources required scale exponentially with the size of the input. In fact, just storing the state of n-particles requires exponential memory. This was the reason which prompted Feynman to conjecture that, if we were to build a controllable quantum system, then perhaps we will be able to use the quantum properties of the controllable quantum system to learn more about the quantum system in which we are interested. 

Feynman's conjecture has come very close to reality. Quantum hardware has made tremendous progress over the last two or three decades. The coherence times of the quantum systems have been steadily increasing and so has been the fidelity of the quantum gates. This has also inspired lot of theoretical developments in the field of quantum simulation. With all these developments, we are within years of making Feynman's vision a reality! 

In this notebook, we will explore the different aspects of quantum simulation. This notebook is written for someone who already has decent exposure to quantum chemistry and knows quantum computing at least to the level presented in Qubes course.

<img src="images/qsim-graphic.gif"  height="600" width="700" />

> <font color=b64c35> The goal of quantum simulation is to study quantum systems using a quantum computer.</font>

The quantum systems could consist of distinguishable quantum particles, or indistinguishable quantum particles like Fermions or Bosons. Further, the purpose of quantum simulation could vary as well. We could be interested in the ground state energy of the system, or we may want to simulate a large system in order to study phase transition. Based on the system you want to study and the properties of the system that you are interested in, the algorithms you use change. 

To demonstrate the various concepts and steps involved in quantum simulation, we will consider a fermionic system. Fermionic systems are of special interest because of problems in quantum chemistry, and material science. 

<h1 style="color:MediumSeaGreen;"> Fermionic quantum simulation </h1>

As mentioned above, one of the goals of quantum simulation is to look for the solutions of the <font color=b64c35>Schrodinger equation:</font>
    
\begin{eqnarray}
i\frac{\partial \psi}{\partial t} (\vec{\mathbf{x}},t)
&=&
H(\vec{\mathbf{x}})
\psi(\vec{\mathbf{x}},t)
\end{eqnarray}
where, for a fermionic system with $N$-electrons and $M$- nuclei, the non-relativistic and time independent Hamiltonian in atomic units is given by:
\begin{align}
\mathscr{H}=&-\sum_{i=1}^{N}\frac12\nabla_i^2-\sum_{A=1}^{M}\frac{1}{2M_A}\nabla_A^2-\sum_{i=1}^N\sum_{A=1}^{M}\frac{Z_A}{r_{iA}}+\\
& \sum_{i=1}^{N}\sum_{j>i}^{N}\frac{1}{r_{ij}}+\sum_{A=1}^{M}\sum_{B>A}^{M}\frac{Z_AZ_B}{r_{AB}}
\end{align}
where, $r_{xy}=|r_x-r_y|$ is the distance between $x^{th}$ electron (or nuclei) and $y_{th}$ electron (or nuclei), $Z_A$ refers to the atomic number of the nuclei, $\nabla_i^2$ and $\nabla_A^2$ are laplacian operators for electrons and nuclei, respectively. The convention is to use lower case indices for the electrons and upper case indices for nuclei. 

The first two terms represent the kinetic energy of electrons and nuclei. The third term represents the potential energy due to Coloumb attraction between electrons and nuclei. The fourth and fifth terms represent the potential energy due to Coloumb repulsion among electrons and nuclei respectively.



<font color=b64c35 size=4>Born-Oppenheimer Approximation</font>

In our presentation in this notebook we will just be dealing with quantum chemistry systems like atoms and molecules. The Born-Oppenheimer approximation is very important to the approach taken in quantum chemistry for solving Schrodinger's equation. It allows us to decouple the dynamics of nuclei from the dynamics of electrons. Qualitatively, it says that since the nuclei are much heavier compared to electrons, they move much slowly compared to the electrons and hence can be assumed to be stationary. This lets us to ignore the second term (kinetic energy of nuclei) and the last term (repulsion between the nuclei) in the Hamiltonian. This effective Hamiltonian is called the electronic Hamiltonian:

\begin{align}
\mathscr{H}_{elec}=&-\sum_{i=1}^{N}\frac12\nabla_i^2-\sum_{i=1}^N\sum_{A=1}^{M}\frac{Z_A}{r_{iA}}+\sum_{i=1}^{N}\sum_{j>i}^{N}\frac{1}{r_{ij}}
\end{align}
Our aim is to look for the eigenfunctions of the above equation. Those eigenfunctions will be N-electron wavefunctions which will depend on the coordinates of electrons and on the coordinates of nuclei, parametrically. Such a wavefunction is called an electronic wavefunction:
$$\mathscr{H}_{elec}\Phi_{elec}(\{r_i\}; \{R_A\})=\mathscr{E}\Phi_{elec}(\{r_i\}; \{R_A\})$$

$\mathscr{E}_{elec}(\{R_A\})$ is the electronic energy which also depends parametrically on the nuclear coordinates. To get the total energy, the kinetic energy of the nuclei and the Coulombic replusion energy have to be added to the electronic energy.

Once we have the electronic wave function, the motion of the nuclei could be solved by assuming the nuclei to be moving in the field generated by the electrons. This lets us generate the potential energy surface which could then be used to calculate the vibrational and rotational modes of the atom/molecule.


<font color=b64c35 size=4>Pauli exclusion principle or the anti-symmetric nature of the electronic wave function </font>

As mentioned earlier, electrons are indistinguishable particles, which means that the complete wave function describing $N$-electrons needs to be either symmetric or anti-symmetric under the exchange of two electrons. Electrons are Fermions with spin $\frac12$, so the allowed $N$-electron wave function needs to be anti-symmetric in nature. The anti-symmetric requirement is with respect to the interchange of both spatial and spin coordinates of any two electrons.

The spin of the electron in non-relativistic context is introduced using two spin functions $\alpha(\omega)$ and $\beta(\omega)$, where $\omega$ is an unspecfied spin variable. The two spin functions are orthonormal. The spin variable $\omega$ along with the spatial variable $r$, are denoted by $\vec{\mathbf{x}}$. The $N$-electron wavefunction depends on $\{\vec{\mathbf{x_1}},\vec{\mathbf{x_2}}...\vec{\mathbf{x_N}}\}$

<font color=b64c35 size=4>Orbitals and Slater determinants</font>

As previously mentioned that the analytical solution to the electronic Hamiltonian is very difficult to come by. Even the numerical methods are complicated by the requirement that the wavefunction must be anti-symmetric in nature. The way the solution is approached is by starting with single particle wavefunctions. These are called orbitals. In the context of atoms and molecules, these are called atomic orbitals and molecular orbitals respectively. These orbitals are functions of the spatial coordinates $\mathbf{r}$ and are called spatial orbitals. The set of spatial orbitals, $\{\psi_i\}$ are chosen to be orthonormal. If the set is infinite, then the set is also complete. In general, the truncated set of spatial orbitals is used. The single electron wavefunction also needs to take into account the spin. So, two spin orbitals are constructed for each spatial orbital using the two orthonormal spin functions, $\alpha(\omega)$ and $\beta(\omega)$. Each spatial orbital is multiplied with $\alpha$ and $\beta$ corresponding to spin up or down, to get the spin orbitals, $\{\chi(\mathbf{x})\}$

\begin{align}  
\chi(\mathbf{x})= 
     \begin{cases}
       \psi(\mathbf{r})\alpha(\omega)\\
       \\
       \psi(\mathbf{r})\beta(\omega)
     \end{cases}
\end{align}

If the spatial orbitals are othonormal, then the spin orbitals will also be orthonormal. So far we have talked about what properties we would like $\chi(\mathbf{x})$ to have, but we have not commented about how to choose $\chi(\mathbf{x})$ or how to obtain them. If we are allowed to go with the set of orbitals which is infinite, then it does not really matter which spatial functions we choose. But, given the constraints on computational resources, we want to choose spatial orbitals which are a good approximation of the actual orbitals. This allows us to truncate the set to a very small size and achieve the required accuracy. One of the ways we obtain such a set is by solving for the eigenfunctions of the $\mathscr{H}_{elec}$ with the interaction between the electrons ignored. As the interaction term is ignored, the Hamiltonian becomes:  
\begin{align}
\mathscr{H}^{'}_{elec}=&-\sum_{i=1}^{N}\frac12\nabla_i^2-\sum_{i=1}^N\sum_{A=1}^{M}\frac{Z_A}{r_{iA}}
\end{align}
This is an easy problem to solve as the Hamiltonian can be thought of as a sum of single electron 'Hamiltonians'. What we mean is that the total Hamiltonian is just the sum of the kinetic energy operator and the potential energy operator for all the $N-$electrons.
$$\mathscr{H'}_{elec}=\sum_{i=1}^N h(i)$$
So, we could just solve for the single electron wave functions for $h(i)$ and construct the $N$-electrons wavefunction from all single electron wavefunctions. We can solve $h(i)$ to get a set of single electron wavefunctions $\{\chi_i\}$. From these, we can form N-electron wavefunctions by taking products of $N$ single electron functions. 
$$\Psi^{t}_{prod}(x_1,x_2...x_N)=\chi_i(x_1)\chi_j(x_2)...\chi_p(x_N)$$
It can be shown that these N-electron wavefunctions are in fact the eigenfunctions of $\mathscr{H'}_{elec}$
$$\mathscr{H}^{'}_{elec}\Psi_{prod}=\mathscr{E}^{'}_{elec}\Psi_{prod}$$
Further, it can be show that $\{\Psi^{t}_{prod}\}$ forms a basis for the $N$-electron wavefunctions.


There is one important property that any valid Fermioinic wavefunction must have and something that $\Psi^t$ (moving forward we will drop the `prod` subscript)  does not have: anti-symmetry. Any valid N-electron wavefunction must be anti-symmetric in nature. The way we can make $\Psi_t$ antisymmetric is to generate all the permutations $\chi_i(x_2)\chi_j(x_1)...\chi_p(x_N)$, $\chi_i(x_2)\chi_j(x_N)...\chi_p(x_1)$... and couple them with the appropriate sign. This can be done with the $sgn(\sigma)=(-1)^{n(\sigma)}$ function. Starting from the base configuration, $n(\sigma)$ tells how many swaps are required to get to the required permutation $\sigma$.
We will denote the anti-symmetrized N-electron wavefunction by $\left|\Psi^t (x_1,x_2..x_N)\right\rangle$. The same expression could also be generated using the following determinant:
\begin{align}
\left|\Psi^t (x_1,x_2..x_N)\right\rangle=\frac{1}{\sqrt{N!}}\begin{vmatrix}
\chi_i(x_1) & \chi_j(x_1) & \cdots & \cdots & \chi_p(x_1)\\
\chi_i(x_2) & \chi_j(x_2) & \cdots & \cdots& \chi_p(x_2)\\
\vdots & \vdots & \cdots & \cdots & \cdots\\
\chi_i(x_N) & \chi_j(x_N) & \cdots & \cdots & \chi_p(x_N)\\
\end{vmatrix}
\end{align}
where $\sqrt{N!}$ is the normalization factor. This determinant is called the Slater determinant, which is named after John C Slater, who introduced it in 1929 as a way to ensure the anti-symmetric nature of a many-electron system. 


<font color=b64c35 size=4>Hartree-Fock approximation</font>

The Slater determinants provide us with the basis for the $N$-electron wavefunctions. But, we still need to get the eigenfunctions of the $\mathscr{H}_{elec}$. Recall that the set of single electron wavefunctions, $\{\chi\}$ is infinite. This implies that the number of possible Slater determinant of $N$-electrons is also infinite. In order to make the problem tractable we truncate the set of single electron wavefunction. Assume that we truncate the set to have $M$ functions. Then, the number of possible N-electron wave functions are $\binom{M}{N}$. This number could be huge! Even for a relatively small system with, say, 100 electrons, if you start with 300 single-electron wavefunctions, this number is bigger than the number of atoms in the universe. And this is the reason why solving problems in quantum chemistry is computationally inefficient. So, most of the problems are approached with approximation techniques. Over the past decades, lot of approximation techniques have been developed. All these techniques are sort of confluence of art and science. Lots of intuition along with the scientific knowledge of the system is required to come up with the best approximation technique which would let us understand a system better.

One of the simplest approximations, which was developed by D.R. Hartree and improved by V.A Fock, is called the Hartree-Fock approximation. The method to obtain the eigenfunction of $\mathscr{H}_{elec}$ using Hartree-Fock approximation is called the Hartree-Fock method. The detailed discussion of the Hartree-Fock method is beyond the scope of this notebook. Here, we will qualitatively describe Hartree-Fock methods and the details will be covered in the crash course on quantum simulation.

Recall that the total number of possible N-electron wavefunctions are $M \choose N$. So, the eigenfunction of $\mathscr{H}_{elec}$ in general is a linear combination of all the N-electron basis functions. The Hartree-Fock method approximates the eigenfunction using a single Slater determinant. The qualitative description of the Hartree-Fock method to calculate the ground state energy of the Hamiltonian is given below.


<font color=b64c35 size=4>Hartree-Fock method</font>

1. Starting with $\mathscr{H}_{elec}^{'}$ (the electronic Hamiltonian with the interaction terms ignored.), solve for $\{\chi_i\}$. 
2. Recall, $\chi_i$ is the eigenfunction of $h(i)$ (kinectic energy and potential energy operator for an electron). Calculate eigenvalues (energies), corresponding to each $\chi_i$.
3. Choose the N-orbitals with the minimum energy and calculate their Slater determinant to get the anti-symmetrized N-electron wavefunction.
4. Using the N-electron wavefunction calculate the Hartree-Fock potential $V_{effective}$. This step is the essence of the Hartree-Fock approximation (another important assumption that we have already mentioned is using a single Slater determinant for approximating the N-electron wavefunction). The idea is to replace the complicated potential of N electrons by a field. 
5. This $V_{effective}$ can be added to $h(i)$ and new eigenfunctions $\chi_i$ can be calculated.
6. The whole procedure is repeated until the solution coverges. Due to this, Hartree-Fock is called a self-consistent method. 

Hartree-Fock method is a surprisingly effective approximation for many systems, and this is why it is the go-to starting point of many techniques in quantum chemistry. 
Detailed discussion about Hartree-Fock and how to use it as a starting point is relegated to the crash course.

<font color=b64c35 size=4>Quantum Chemistry Software</font>

A lot of research in quantum chemistry has led to development to a lot of very good software, which runs calculations like Hartree-Fock and other self-consistent methods. Some of these software are: pyscf, Gaussian, psi4, pyquante. A detailed list can be found [here](https://en.wikipedia.org/wiki/List_of_quantum_chemistry_and_solid-state_physics_software). pyscf and psi4 are very popular in the quantum computing community because they are open-source. pyqante is also opensource but it is does not have as many features as pyscf and psi4. In this notebook we will be using pyscf. Following is the code snippet for running a Hartree-Fock calculation for $H_2$ molecule.

In [ ]:
from pyscf import gto
mol = gto.Mole()
mol.verbose = 5
mol.output = 'h2.log'
mol.atom = 'H 0 0 0; H 0 0 1'
mol.basis = 'sto-3g'
mol.build()
from pyscf import scf
m = scf.RHF(mol)
print('E(HF) = %g' % m.kernel())

<h2 style="color:MediumSeaGreen;"> Why do we need quantum computation for quantum chemistry? </h2>
We would like to remind you that the Hartree-Fock method is an approximation and the results we obtain while calculating properties of molecules using the Hartree-Fock method does not compare well with the experiments. You can definitely build upon Hartree-Fock with other methods, which are much more accurate than Hartree-Fock, but which scale exponentially. This is where quantum computing comes in. Let's switch gears and talk about how to approach quantum chemistry problems using quantum computing.

<h2 style="color:MediumSeaGreen;"> Phase estimation algorithm </h2>

There are different types of quantum algorithms which have been shown to be better than their classical counterparts. Many of the applications have come out of just starting with one such algorithm and applying it to solve a problem of interest. The phase estimation algorithm is one such algorithm which can be used to calculate the ground state of a quantum Hamiltonian. Let's briefly review the phase estimation algorithm (PEA).

The problem that PEA addresses is the following:

> <font color=b64c35 size=4> Given a unitary $U$ and its eigenvector $\left|\psi\right\rangle$ with an eigenvalue $e^{2\pi i \phi}$, find $\phi$. </font>

The PEA employs two quantum registers. A quantum register is a system of multiple qubits. The first quantum register will be used to read out binary representation of $\phi$. The number of qubits in the first register is dictated by the required precision in the value of $\left|\phi\right\rangle$. The second register is used for preparing the eigenstate $\left|\psi\right\rangle$ of $U$, and implementing controlled-$U^{2^j}$ for certain positive powers of $j$. Therefore, the number of qubits in the second register will depend on the number of qubits required by $\psi$ and $U$.

The algorithm proceeds as follows:

1. The first step is to create an equal superposition of all the basis states on the first register of $t$-qubits, as shown in the circuit above.  This is done by applying Hadamard gates on the first $t$-qubits in $\left|0\right\rangle$ state. Interestingly, this is also the QFT of all zeros states.

$$H^{\otimes t}\left|0\right\rangle^{\otimes t}\left|\psi\right\rangle=\frac{1}{2^{t/2}}(\left|0\right\rangle_{0}+\left|1\right\rangle_1)\otimes(\left|0\right\rangle_1+\left|1\right\rangle_1)\otimes ...(\left|0\right\rangle_{t-1}+\left|1\right\rangle_{t-1})\left|\psi\right\rangle=\frac{1}{2^{t/2}}\sum_{j=0}^{2^t-1}\left|j\right\rangle \left|\psi\right\rangle$$

2. The second step is to successively apply controlled-$U^{2s}$ operations on the second register, as shown in the circuit above. The state of the qubits is given as:

$$\frac{1}{2^{t/2}}(\left|0\right\rangle_{0}+e^{2\pi i 2^0 \phi}\left|1\right\rangle_0)\otimes(\left|0\right\rangle_1+e^{2\pi i 2^1 \phi}\left|1\right\rangle_1)\otimes ...(\left|0\right\rangle_{t-1}+e^{2\pi i 2^{t-1} \phi}\left|1\right\rangle_{t-1})\left|\psi\right\rangle$$

3. Apply $QFT^{-1}$, to get the binary representation of $\phi$ as the output.

One interesting observation for phase estimation algorithm is that, even if we know the eigenfunction approximately, we can still recover the eigenvalue with good probability.

This is should give you a hint as to how the phase estimation algorithm may be used for quantum chemistry applications. It turns out, if we choose our unitary, $U$ to be $U=e^{-i\mathscr{H}_{elec}t}$, then we can use ev
en the approximation of the ground state and, with good probability, we will recover the exact ground state energy (in a given basis). This approximation of the ground state comes from the Hartree-Fock method. We just described to you, qualitatively, the big picture of doing quantum chemistry on a quantum computer. The details of mapping $\mathscr{H}_{elec}$ to qubit operators and initializing the approximate ground state obtained from the Hartree-Fock method, in terms of qubits, are given below.

<h2 style="color:MediumSeaGreen;"> Second quantization </h2>
Before we move on to representing the electronic Hamiltonian in terms of qubit operators, let us describe to you the second quantization formalism. The Slater determinant allows us to make the N-electron system anti-symmetric, but the expression that we get from the Slater determinant is cumbersome to deal with. It gets even worse when working with a general N-electron state and its Hamiltonian to get its expectation value. 

Second quantization encodes the anti-symmetric property of the wavefunction in the algebraic operators. This also simplifies the representation of the electronic Hamiltonian. We will start with Slater determinant notation, introduce the second quantization operators, and show how the Slater determinant properties are encoded in the algebraic properties of these operators.

Let's assume that our truncated set of orbitals, $\{\chi_i\}$ consists of $M$ modes. Then an $N$-electron Slater determinant is represented as $\left|\chi_i(x_1)\chi_j(x_2)...\chi_p(x_N)\right\rangle$. We define a slightly different notation, where we fix the order of the coordinates and hence drop them, and represent the Slater determinant with a binary string of length $M$, i.e. $\left|00..0\right\rangle$. If a particular mode, say $i^{th}$ mode, is present then that is represented using a 1 in $i^{th}$ position. So the slater determinant $\left|\chi_i(x_1)\chi_j(x_2)...\chi_p(x_N)\right\rangle$ is represented using $\left|0.1..0..1..1..0\right\rangle$ where 1's are in position $i,j,..p$. This is also called occupation number basis representation. In order to create an $N+1$ or $N-1$ electron state from an $N$-electron state, we define fermionic creation and annihilation operators, $a^{\dagger}_i$ and $a_i$. These act on the state as follows:
\begin{align}
a_j^{\dagger} \left|f_1...f_{j-1} 0 f_{j+1}...f_{M}\right\rangle&= (-1)^{\Gamma_j}\left|f_1...f_{j-1} 1 f_{j+1}...f_{M}\right\rangle\nonumber\\
a_j^{\dagger} \left|f_{1}...f_{j-1} 1 f_{j+1}...f_M\right\rangle &= 0\nonumber\\
a_j \left|f_{1}...f_{j-1} 1 f_{j+1}...f_M\right\rangle &= (-1)^{\Gamma_j}\left|f_{1}...f_{j-1} 0 f_{j+1}...f_M\right\rangle\nonumber\\
a_j \left|f_{1}...f_{j-1} 0 f_{j+1}...f_M\right\rangle &=0
\end{align}

Here $\Gamma_j=\sum_{s=1}^{j-1}f_s$, and $f_{i}$ represents the occupation of fermionic mode $\chi_{i}$. We now define the vacuum state as $\left|\Omega\right\rangle$ with no fermionic particles present. We can represent the N-electron state as:
$$a^{\dagger}_ia^{\dagger}_j...a_p^{\dagger}\left|\Omega\right\rangle\equiv \left|\chi_i\chi_j...\chi_p\right\rangle$$
The order in which the the creation operators are applied is important. The creation operators are applied sequentially and with each application of a creation operator, the state will pick up a phase $\sum_{s=1}^{j-1}f_s$. So, the overall phase depends on the order in which the creation operators are applied.



Based on the definitions of the fermionic creation and annihilation operators, we can deduce their algebra to be:

$$a^{\dagger}_ia_j+a_ja^{\dagger}_i=\delta_{ij} \qquad a_ia_j+a_ja_i=0 \qquad a^{\dagger}_ia^{\dagger}_j+a^{\dagger}_ja^{\dagger}_i=0$$

These are called cannonical anti-commutation relations. These will be very usefull when we are dealing with Hamiltonian and the general $N$-electron wavefunction. It can be shown that the Hamiltonian has the following representation in the second quantization formalism:
\begin{align}
H=\sum_{ij}^M h_{ij}a_{i}^{\dagger}a_{j}+\frac{1}{2} \sum_{ijkl}^M h_{ijkl}a_{i}^{\dagger}a_{j}^{\dagger}a_{k}a_{l}
\end{align}
where $h_{ij}$ and $h_{ijkl}$ are one-electron and two-electron integrals defined as:
\begin{align}
h_{ij} &=\int\chi_{i}^{\ast}(x,\omega)\left(- \frac{\nabla^2_x}{2} +V_{ext}(x)\right)\chi_{j}(x,\omega)dx d\omega \nonumber \\
& =\delta_{\sigma_{i}\sigma_{j}}\int\phi_{i}^{\ast}(x) \left(- \frac{\nabla^2_x}{2} +V_{ext}(x)\right)\phi_{j}(x)dx \end{align}
and
\begin{align}
h_{ijkl} 
&=\delta_{\sigma_{i}\sigma_{l}}\delta_{\sigma_{j}\sigma_{k}} \int\frac{\phi_{i}^{\ast}(x_{1})\phi_{j}^{\ast}(x_{2})\phi_{k}(x_{2})\phi_{l}(x_{1})}{|x_{1}-x_{2}|} dx_{1}dx_{2} 
\end{align}
Here, the modes within the basis set were decomposed as $\chi_i(x,\omega)=\phi_i(x)\sigma_i(\omega)$ with $\phi_i(x)$ as the spatial orbital and $\sigma_i(\omega)$ as the spin function. Since the Hamiltonian does not interact with the spin component of the wave function, $\sigma_i$ is either $\alpha(\omega)$ or $\beta(\omega)$ corresponding to spin up or spin down. 

<h2 style="color:MediumSeaGreen;"> Fermionic Encodings </h2>

In order to use the phase estimation algorithm to get the ground state energy of the electronic Hamiltonian, we need to get the qubit operator representation of the Hamiltonian. There are many mappings/transformations/encodings that map the fermionic creation and annihilation operators to the qubit operators. The creation/annihilation operators in the Hamiltonian can then be replaced with their qubit operator representation to get the qubit operator representation of the whole Hamiltonian. 

Before we present to you the details of these encodings, let us go over the algebra of qubit operators. For a single qubit, the Pauli, X ($\sigma_x$), Y ($\sigma_y$), Z ($\sigma_z$) and I matrices form the basis for representing any unitary matrix. Similarly, the Pauli group over N-qubits along with I forms the basis for representing any unitary over N qubits. Pauli matrices obey the following commutation relations:

$$[\sigma_p,\sigma_q]=2\epsilon_{pqr}\sigma_r \qquad \{\sigma_p,\sigma_q\}=2\delta_{pq}I$$

Fermionic encodings aim to encode the fermionic states and operators in terms of qubit states and operators respectively. It is important to have efficient encodings, as this would influence the efficiency in implementing the unitary $U$ for the phase estimation algorithm. Many encodings have been proposed over the past decades which improve the scaling. In this notebook, we will cover Jordan-Wigner, parity and Bravyi Kitaev. These are the most commonly followed in the literature and are relatively easy to understand. Jordan-Wigner encoding will be discussed in detail, whereas the discussion of parity and Bravyi-Kitaev mapping will be succinct and at qualitative level.

<font color=b64c35 size=4>Jordan-Wigner</font>

In Jordan-Wigner encoding, the occupation number of a fermionic mode is stored in the state of the qubit. e.g. if $i^{th}$ mode is occupied then the state of the $i^{th}$ qubit will be represented $\left|1\right\rangle$. The unoccupied fermionic mode is mapped to the state $\left|0\right\rangle$ of the qubit. The N-electron state is mapped to N-qubits as follows:

$$\left|011...1...0\right\rangle_{elec} \longrightarrow \left|0\right\rangle\otimes\left|1\right\rangle\otimes\left|1\right\rangle\otimes...\left|1\right\rangle\otimes...\left|0\right\rangle\otimes$$

Coming up with the correct qubit operators for the fermionic operators is little more tricky. From our knowledge of spin operators, we know that the raising and lowering operators are given by:
\begin{align}
Q^{\pm}=\frac{1}{2}(\sigma^{x}\mp i\sigma^{y})
\end{align}

If we were dealing with single fermionic mode that these raising and lowering operators could be mapped to creation and annihilation operators. But, with multiple fermionic modes in the system, observe that mapping creation and annihilation operators to raising and lowering operators does not take into account the phase factor, $(-1)^{\Gamma_j}$, for the fermionic states.

The following qubit operators obey the fermionic algebra:
\begin{align}
a_j^{\dagger}\equiv\mathbf{1}^{\otimes n-j-1}\otimes Q_{j}^{+}\otimes[\sigma^z]^{\otimes j}\\
a_j\equiv\mathbf{1}^{\otimes n-j-1}\otimes Q_{j}^{-}\otimes[\sigma^z]^{\otimes j}
\end{align}
Analyzing the above expressions we recognize that the implementation of fermionic creation and annihilation operators  costs $O(M)$ qubit operators, where $M$ is the number of modes.

<font color=b64c35 size=4>Parity mapping</font>

In the Jordan-Wigner mapping, the main cost of fermionic operators is due to the string of $\sigma_z$ operators (which computes the parity) that precede the raising and lowering operators. So, if we were to number the fermionic modes and the qubits, and in each qubit store the parity of all the preceding qubits then we would avoid the $O(M)$ $\sigma_z$ gates to calculate the parity. But in the parity mapping we end up spending $O(M)$ $\sigma_x$ gates to update the parity of the succeding qubits. So we do not actually improve the efficiency of implementing fermionic operators.
 
<font color=b64c35 size=4>Bravyi-Kitaev</font>

Bravyi-Kitaev is a compromise between the Jordan-Wigner mapping and the parity mapping. It stores the occupation number of the fermionic modes in some of the qubits and partity of qubits in some other qubits. This strategy lets the fermionic operators be implemented in $O(log(M))$ qubit operators.

<h2 style="color:MediumSeaGreen;"> Trotter-Suzuki Approximation </h2>

The fermionic encodings lets us transform the second quantization representation of the Hamiltonian to the qubit operator representation of the Hamiltonian. 

\begin{align}
H=\sum_{ij}^M h_{ij}a_{i}^{\dagger}a_{j}+&\frac12 \sum_{ijkl}^M h_{ijkl}a_{i}^{\dagger}a_{j}^{\dagger}a_{k}a_{l} \quad \longrightarrow  \quad H = \sum_{j=1}^{r}\boldsymbol{\eta_j} \\ \label{hyd_ham}
\boldsymbol{\eta_j}\in &\boldsymbol{P}\quad (\text{Pauli group})
\end{align}

As we discussed, in order to retrieve the ground state energy of the we would use PEA and in order to use PEA, we need to implement 

$$U=e^{-iHt}=e^{-i\left(\sum_{ij}^M h_{ij}a_{i}^{\dagger}a_{j}+\frac12 \sum_{ijkl}^M h_{ijkl}a_{i}^{\dagger}a_{j}^{\dagger}a_{k}a_{l}\right)t}=e^{-i\left(\sum_{j=1}^{r}\boldsymbol{\eta_j}\right)t}$$

Since, in general $\eta_i$s do not commute with each other we cannot have
$$e^{-i\left(\sum_{j=1}^{r}\boldsymbol{\eta_j}\right)t}=\prod_{j=1}^re^{-i\eta_jt}$$
However, it still is a very good first order approximation. This is also called the Trotter-Suzuki approximation. Masuo Suzuki and Hale Trotter worked on quantifying the errors from such approximations and on developing higher order formulae to reduce the error.

Using the Trotter-Suzuki approximation each term, $e^{-i\eta_jt}$ can be successively created to implement the unitary evolution under the electronic Hamiltonian. One last place we need to use PEA is the initial state. Since we are using Jordan-Wigner encoding which encodes the occupation number of fermionic modes in qubits, the Hartree-Fock state of fermionic system with $M$-modes and $N$-electrons is just $\left|11..10...00\right\rangle$. This is represented with the first $N$ qubits in state $\left|1\right\rangle$ and the rest of $M-N$ qubits in state $\left|0\right\rangle$.



<h1 style="color:MediumSeaGreen;"> Implementation of quantum simulation in qiskit-chemistry and openfermion</h1>

Quantum simulation has a high entry barrier. This is because it requires understanding of so many different domains. If some of the concepts presented above are not very clear then that is okay. The purpose of this notebook is to give you an enough overview of the field of quantum chemistry, so that you can understand various steps involved in quantum simulation.

In this section we will talk about simulating a very small quantum chemistry system like a $H_2$ molecule using qiskit and openfermion. qiskit-chemistry (by IBM) and openfermion (by Google) are the two open-source software packges for doing quantum chemistry on quantum computers. 

As we go through the code, you will observe that the simulation could be structured into three parts:

1. Setting up the system in pyscf and running Hartree-Fock calculation. This also generates the integrals for the Hamiltonian ($h_{ij}$,$h_{ijkl}$).
2. Using the integrals obtained from pyscf, to transform the Hamiltonian from second quantization formalism to qubit operators.
3. Using the transformed qubit operator to get the ground state energy using one of the various algorithms: phase estimation algorithm (PEA), variational quantum eigensolver (VQE), exact-diagonalization, etc.

For the step 3, we will using exact diagonalization to get the ground state energy. Exact diagonalization is a very inefficient process and does not use quantum computers. It is used for small systems as a way to verify results from quantum computers. e.g. in this case, it also code for pyscf and for Jordan-Wigner transformation. Using phase estimation algorithm would require many more qubits and much bigger quantum circuit. This would add to the complexity of the simulation and hence is relegated to the detailed course on quantum simulation.


<font color=b64c35 size=5>**Qiskit-chemistry**</font>


In [37]:
# Importing the packages
import numpy as np
# Importing the driver that lets us access pyscf from qiskit
from qiskit.chemistry.drivers import PySCFDriver
# Importing the function for exact diagonalization
from qiskit.aqua.algorithms import NumPyEigensolver as EE
# Import the class fermionicoperator
from qiskit.chemistry import FermionicOperator

# 1. Setting up the system in pyscf and running Hartree-Fock calculation.
# Define the molecule
molecule = 'H .0 .0 0.0;H .0 .0 0.7414'
# Setup the pyscf driver in qiskit
driver = PySCFDriver(atom=molecule, basis='sto3g')
# Run the pyscf driver
qmolecule = driver.run()
# get the one-body and two-body integrals
one_b = qmolecule.one_body_integrals
two_b = qmolecule.two_body_integrals


# # 2.Using the integrals obtained from pyscf, to transform
# # the Hamiltonian from second quantization formalism to qubit operators.
fer_op = FermionicOperator(h1=one_b, h2=two_b)
# # transform the fermionic operator to qubit operator
qubit_op = fer_op.mapping('jordan_wigner')



# # 3. exact diagonalize the matrix form of the qubit operator
result = EE(qubit_op).run()
# lines, result = operator.process_algorithm_result(result)
energies = result['eigenvalues']

# print('One body integrals:', one_b)
# print('Two body integrals:', two_b)
print(qubit_op.print_details())
# print('Distance: ', 0.7414)
# print('Energy:', energies)

IIII	(-0.8126179630230757+0j)
IIIZ	(0.17119774903432938+0j)
IIZI	(-0.22278593040418493+0j)
IZII	(0.1711977490343293+0j)
ZIII	(-0.2227859304041849+0j)
IIZZ	(0.12054482205301804+0j)
IZIZ	(0.1686221915892094+0j)
XXYY	(0.045322202052873996+0j)
YYYY	(0.045322202052873996+0j)
XXXX	(0.045322202052873996+0j)
YYXX	(0.045322202052873996+0j)
ZIIZ	(0.16586702410589205+0j)
IZZI	(0.16586702410589205+0j)
ZIZI	(0.17434844185575674+0j)
ZZII	(0.12054482205301804+0j)



<font color=b64c35 size=5>**OpenFermion**</font>


In [38]:
# Importing the relavant classes and functions
from openfermion.hamiltonians import MolecularData
from openfermion.transforms import get_fermion_operator, get_sparse_operator, jordan_wigner
from openfermion.utils import get_ground_state

# 1. Setting up the system in pyscf and running Hartree-Fock calculation.
# Define the molecule
diatomic_bond_length = .7414
geometry = [('H', (0., 0., 0.)), ('H', (0., 0., diatomic_bond_length))]
basis = 'sto-3g'
multiplicity = 1
charge = 0
description = str(diatomic_bond_length)

# Initialize the molecule
molecule = MolecularData(geometry, basis, multiplicity,
                         charge, description)
# Just like qiskit-chemistry openfermion uses pyscf, but the calculation is done using
# a plugin and the results are stored and can be used later. The results for H2 are
# already present, so we can just use them.
molecule.load()

In [39]:

# Get the Hamiltonian in an active space.
molecular_hamiltonian = molecule.get_molecular_hamiltonian()

# 2.Using the integrals obtained from pyscf, to transform
# the Hamiltonian from second quantization formalism to qubit operators.
fermion_hamiltonian = get_fermion_operator(molecular_hamiltonian)
qubit_hamiltonian = jordan_wigner(fermion_hamiltonian)
qubit_hamiltonian.compress()
print('The Jordan-Wigner Hamiltonian in canonical basis follows:\n{}'.format(qubit_hamiltonian))

# 3. exact diagonalize the matrix form of the qubit operator
sparse_hamiltonian = get_sparse_operator(qubit_hamiltonian)
energy, state = get_ground_state(sparse_hamiltonian)
print('Ground state energy before rotation is {} Hartree.\n'.format(energy))


The Jordan-Wigner Hamiltonian in canonical basis follows:
-0.09886397351781583 [] +
-0.04532220209856541 [X0 X1 Y2 Y3] +
0.04532220209856541 [X0 Y1 Y2 X3] +
0.04532220209856541 [Y0 X1 X2 Y3] +
-0.04532220209856541 [Y0 Y1 X2 X3] +
0.17119774853325848 [Z0] +
0.16862219143347554 [Z0 Z1] +
0.12054482186554413 [Z0 Z2] +
0.16586702396410954 [Z0 Z3] +
0.1711977485332586 [Z1] +
0.16586702396410954 [Z1 Z2] +
0.12054482186554413 [Z1 Z3] +
-0.22278592890107018 [Z2] +
0.17434844170557132 [Z2 Z3] +
-0.22278592890107013 [Z3]
Ground state energy before rotation is -1.1372701746253266 Hartree.



In [2]:
from qbraid_chem_widget import ChemWidget
ChemWidget()

ChemWidget(jupyterhub_username='kanavsetia1@gmail.com')

In [3]:
import numpy as np
from qiskit.chemistry.drivers import PySCFDriver
from qiskit.aqua.algorithms import ExactEigensolver as EE
from qiskit.chemistry import FermionicOperator
molecule = 'H .0 .0 0.0;H .0 .0 0.7414'
driver = PySCFDriver(atom=molecule, basis='sto3g')
qmolecule = driver.run()